In [50]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
import gradio as gr

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

#embeddings = model.encode(sentences)

#similarities = model.similarity(embeddings, embeddings)
#print(similarities.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1199.14it/s]


In [51]:
df = pd.read_csv('prod.csv')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52955 entries, 0 to 52954
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           52955 non-null  int64  
 1   CustomerID           52924 non-null  float64
 2   Gender               52924 non-null  object 
 3   Location             52924 non-null  object 
 4   Tenure_Months        52924 non-null  float64
 5   Transaction_ID       52924 non-null  float64
 6   Transaction_Date     52924 non-null  object 
 7   Product_SKU          52924 non-null  object 
 8   Product_Description  52924 non-null  object 
 9   Product_Category     52955 non-null  object 
 10  Quantity             52924 non-null  float64
 11  Avg_Price            52924 non-null  float64
 12  Delivery_Charges     52924 non-null  float64
 13  Coupon_Status        52924 non-null  object 
 14  GST                  52924 non-null  float64
 15  Date                 52924 non-null 

In [52]:
products = df.drop_duplicates(subset=['Product_SKU', 'Product_Category', 'Product_Description']).copy()

products['search_text'] = products['Product_Category'] + " " + products['Product_Description']

products['search_text']=products['search_text'].fillna("")#IMPORTANT

product_descriptions = products['Product_Description'].tolist()
search_texts = products['search_text'].tolist()

In [53]:
products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1153 entries, 0 to 52951
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           1153 non-null   int64  
 1   CustomerID           1145 non-null   float64
 2   Gender               1145 non-null   object 
 3   Location             1145 non-null   object 
 4   Tenure_Months        1145 non-null   float64
 5   Transaction_ID       1145 non-null   float64
 6   Transaction_Date     1145 non-null   object 
 7   Product_SKU          1145 non-null   object 
 8   Product_Description  1145 non-null   object 
 9   Product_Category     1153 non-null   object 
 10  Quantity             1145 non-null   float64
 11  Avg_Price            1145 non-null   float64
 12  Delivery_Charges     1145 non-null   float64
 13  Coupon_Status        1145 non-null   object 
 14  GST                  1145 non-null   float64
 15  Date                 1145 non-null   objec

In [54]:
print(search_texts)

['Nest-USA Nest Learning Thermostat 3rd Gen-USA - Stainless Steel', 'Nest-USA Nest Cam Outdoor Security Camera - USA', 'Nest-USA Nest Protect Smoke + CO White Battery Alarm-USA', 'Nest-USA Nest Cam Indoor Security Camera - USA', 'Nest-USA Nest Protect Smoke + CO White Wired Alarm-USA', 'Nest-USA Nest Protect Smoke + CO Black Battery Alarm-USA', 'Nest-USA Nest Protect Smoke + CO Black Wired Alarm-USA', 'Office Google Laptop and Cell Phone Stickers', 'Office Maze Pen', 'Office Galaxy Screen Cleaning Cloth', 'Office Badge Holder', 'Office Ballpoint LED Light Pen', 'Office Color Changing Grip Pen', 'Office Colored Pencil Set', 'Office Spiral Notebook and Pen Set', 'Office Android Sticker Sheet Ultra Removable', 'Office YouTube Custom Decals', 'Office Switch Tone Color Crayon Pen', 'Office Leatherette Journal', 'Office Google Doodle Decal', 'Office YouTube Leatherette Notebook Combo', 'Office Red Spiral Google Notebook', 'Office 8 pc Android Sticker Sheet', 'Office Retractable Ballpoint Pen

In [55]:

embeddings = model.encode(search_texts, show_progress_bar=True)


Batches: 100%|██████████| 37/37 [00:13<00:00,  2.79it/s]


In [56]:
print(embeddings)

[[-0.00934586  0.00718992  0.01375943 ...  0.01751337  0.05151795
   0.04835244]
 [ 0.05060852 -0.03017505  0.01775709 ...  0.02022324 -0.00701634
   0.01615868]
 [-0.0420887  -0.01167906  0.01005798 ...  0.04823035  0.01382341
   0.06368282]
 ...
 [ 0.04846165 -0.00129616  0.01687905 ... -0.02188589  0.03803815
   0.00797058]
 [ 0.04846165 -0.00129616  0.01687905 ... -0.02188589  0.03803815
   0.00797058]
 [ 0.04846165 -0.00129616  0.01687905 ... -0.02188589  0.03803815
   0.00797058]]


In [57]:
#tst.shape

In [58]:
all_embd=[]
for text in search_texts:
    embd = model.encode(text,show_progress_bar=True)
    all_embd.append(embd)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 39.83it/s]


In [60]:
corpus_embeddings = torch.tensor(all_embd)
corpus_embeddings

tensor([[-0.0093,  0.0072,  0.0138,  ...,  0.0175,  0.0515,  0.0484],
        [ 0.0506, -0.0302,  0.0178,  ...,  0.0202, -0.0070,  0.0162],
        [-0.0421, -0.0117,  0.0101,  ...,  0.0482,  0.0138,  0.0637],
        ...,
        [ 0.0485, -0.0013,  0.0169,  ..., -0.0219,  0.0380,  0.0080],
        [ 0.0485, -0.0013,  0.0169,  ..., -0.0219,  0.0380,  0.0080],
        [ 0.0485, -0.0013,  0.0169,  ..., -0.0219,  0.0380,  0.0080]])

In [61]:
def search_similiar(query, n):
    n=int (n)#Enuring it is integer for silicing

    query_embeddings=model.encode(query)
    sims=[]

    for e in corpus_embeddings:

        score=model.similarity(query_embeddings , e).numpy()[0][0]
        sims.append(score)

    # 3. Create a clean copy of the dataframe with the columns you want to show
    # We use products to match the 1,153 embeddings we generated
    df_ind = products[['Product_Category', 'Product_Description']].copy()
    
    # 4. Attach the calculated scores as a new column
    df_ind.loc[:, 'Similarity Score'] = np.array(sims)
    
    # 5. Sort by highest score and return the top 'n' rows
    top_results = df_ind.sort_values('Similarity Score', ascending=False)[:n]
    top_results['Similarity Score'] = (top_results['Similarity Score'] * 100).round(2)
    return top_results
    # Optional: Round the scores for a cleaner display in Gradio
    #top_results['Similarity Score'] = top_results['Similarity Score'].round(4)

In [62]:
search_similiar("Red Blue Tshirt",10)

,Product_Category,Product_Description,Similarity Score
6043,Apparel,Google Tee Blue,77.059998
5995,Apparel,Google Tee Blue,77.059998
6192,Apparel,Google Tee Blue,77.059998
6057,Apparel,Google Tee Blue,77.059998
6131,Apparel,Google Tee Blue,77.059998
6072,Apparel,Google Tee Blue,77.059998
6253,Apparel,Google Tee Blue,77.059998
6091,Apparel,Google Tee Red,76.830002
16467,Apparel,Google Toddler Sports T-shirt Red,75.629997
16716,Apparel,Google Toddler Sports T-shirt Red,75.629997


In [71]:

my_theme=gr.themes.Default(primary_hue="yellow").set(
    block_label_background_fill_dark="transparent",
    block_label_border_width="0px",
    body_background_fill="#0f172a",
    background_fill_secondary="#0f172a"
    )

In [72]:
inter=gr.Interface(
    fn=search_similiar,
    inputs=[
        gr.Textbox(label="Enter a Product Keyword", placeholder="Example: Comfortable Running Shoes"),
        gr.Slider(minimum=1, maximum=20, step=1, value=5,label="Choose Closest Matches (n)")
    ],
    outputs=gr.Dataframe(label="Search Results"),
    title="Product Search Engine",
    description="Type in a keyword to find the closest matching products in your dataset.",
    theme=my_theme
)

e:\AnacondaAnvesh\envs\hf\lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


In [73]:
inter.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
